# Modeling Thoughts

This notebook is meant to test what type of model we might want to use for our data.

## On using Pokemon type as a feature

Logistic regession is not going to be any good at determining the winner of the match if Pokemon types are the primary determining feature of winner and loser.  See empirical data demonstrating this below, but recall that logistic regression assumes that log-odds of being in class 1 (e.g. the log-odds of player 0 winning the match) are linear in the features (e.g. the types of the pokemon in the match).  So a logistic regression using types as features will inherently say something like 'if you add one fire type pokemon to your team, you'll increase your log-odds of winning by 0.9'.  But this doesn't make sense because adding a fire type to your team may increase your odds of winning (if your opponent has lots of grass types) or decrease your odds of winning (if your opponent has lots of water types).  So the logistic regression comes with assumptions (namely that odds of winning are monotonic in the features) which aren't true for us.

What other kinds of models could we use?  Decision trees, random forests, and gradient boosting seems like good options.  From a little bit of reading, random forests and gradient boosting put together a bunch of decision trees, so understanding decision trees seems like a good place to start.  Something I read indicated that gradient boosting is good when data has some missing components, so that could be a good route to go (since we only know all 12 pokemon in a battle about 40% of the time).  However, I have no idea what math and what assumptions lie behind those models, so they may not be of any use.

I have an empirical test for decision trees down below.  It learns rock-paper-scissors perfectly, so I guess that it was made for this kind of thing.  So I think decision tree-related models are likely to be helpful for us.

## On using Pokemon stats as a feature

Even given all of the above, types are not the only features we'll be predicting on.  Pokemon stats (HP, Atk, etc.) are also very relevant.  Unlike types (where having more fire types is not necessarily better), having bigger stats is (in all but very rare circumstances) better.  I would definitely believe that we could run a logistic regression on a dataset whose features are things like 'player_1_pokemon_1_hp' and 'player_2_pokemon_4_attacking_stat' and get very good results.  However, you can't really make synthetic data of this sort, so I have no basis for this claim.

Problematically, there's still some rock-paper-scissors stuff going on with stats.  If your oppenent has a lot of special attackers, then a 1-point increase in defense isn't worth much.  But if your opponent has a lot of physical attackers, then a 1-point increase in defense means more.  But we may still be able to run a logistic regression with an interaction term of the form $\text{Def} \cdot \text{number of opposing physical attackers}$ (where a physical attacker is defined to be a pokemon on the opposing team whose Atk is greater than or equal to its Sp Atk).

## Questions

- What model would be good for predicting outcomes of rock-paper-scissors-like games?  E.g. we want a model that takes a pair like ('rock','scissors') as input and outputs a probability that player 0.
- What model would be good for predicting outcomes of pokemon battles based on stats?  I'd guess logistic regression is actually quite good here.
- Is there a way to "combine" two models into one?  E.g. could we train a decision tree on the types and a logistic regression on the stats and somehow combine those results to predict a winner?

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold,cross_validate
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier

### Logistic Regression

In this section, we'll see (empirically) that a logistic regression does not learn the game rock-paper-scissors very well.  Meaning that if you have a bunch of observations of the form $(\text{p1\_choice},\text{p2\_choice},\text{p0\_wins})$, e.g. $(\text{rock},\text{scissors},1)$, and you train a logistic regression on that data, it will do a poor job of predicting the outcome of a given game.

First, we'll construct some sythetic data.

In [2]:
types = ['fire','water','grass']

matchups = [] # a list of 1000 lists of the form [p1_type, p2_type] where p1_type != p2_type
for i in range(1000):
    leads = []
    leads.append(types[np.random.randint(0,3)]) # pick a random type for p0
    new_type = types[np.random.randint(0,3)]
    while new_type == leads[0]:
        new_type = types[np.random.randint(0,3)]
    leads.append(new_type) # pick a random type for p1 that is not equal to p0's type
    matchups.append(leads)
df = pd.DataFrame(matchups)
target = np.array([int(types.index(matchups[i][0]) == ((types.index(matchups[i][1]) + 1) % 3)) for i in range(len(matchups))]) # 1 if and only player 0 wins

dummy_pipe = Pipeline([
    ('one_hot',OneHotEncoder()),
    ('dummy',DummyClassifier())
])

lr_pipe = Pipeline([
    ('one_hot',OneHotEncoder()),
    ('log_reg',LogisticRegression())
])

scoring = {
    'roc_auc': 'roc_auc',
    'log_loss': 'neg_log_loss',
    'accuracy': 'accuracy'
}

models = [dummy_pipe,lr_pipe]
model_names = ["Dummy classifier", "Logistic regression"]

for i, model in enumerate(models):
    cv_info = cross_validate(model,df,target,cv = StratifiedKFold(),scoring=scoring)
    print(model_names[i] + ":")
    for scoring_key in scoring.keys():
        print("\t" + scoring_key + " mean:", np.mean(cv_info['test_' + scoring_key]))
        print("\t" + scoring_key + " std:", np.std(cv_info['test_' + scoring_key],ddof=1))

Dummy classifier:
	roc_auc mean: 0.5
	roc_auc std: 0.0
	log_loss mean: -0.692429513477528
	log_loss std: 0.00017758133585242436
	accuracy mean: 0.519
	accuracy std: 0.002236067977499792
Logistic regression:
	roc_auc mean: 0.5115002661048791
	roc_auc std: 0.08418708774121862
	log_loss mean: -0.7009857976133204
	log_loss std: 0.010801249405291128
	accuracy mean: 0.5730000000000001
	accuracy std: 0.15405356211396087


We can see that the logistic regression does about as well as the dummy classifier, which is bad!  Let's run the above chunk of code 1000 times and record how often the logistic regression beats the dummy classifier in each of the metrics.

In [3]:
lr_better_roc_auc_count = 0
lr_better_log_loss_count = 0
lr_better_accuracy_count = 0
for j in range(1000):
    matchups = []
    for i in range(1000):
        leads = []
        leads.append(types[np.random.randint(0,3)])
        new_type = types[np.random.randint(0,3)]
        while new_type == leads[0]:
            new_type = types[np.random.randint(0,3)]
        leads.append(new_type)
        matchups.append(leads)
    df = pd.DataFrame(matchups)
    target = np.array([int(types.index(matchups[i][0]) == ((types.index(matchups[i][1]) + 1) % 3)) for i in range(len(matchups))]) # True if and only player 0 wins

    dummy_pipe = Pipeline([
        ('one_hot',OneHotEncoder()),
        ('dummy',DummyClassifier())
    ])

    lr_pipe = Pipeline([
        ('one_hot',OneHotEncoder()),
        ('log_reg',LogisticRegression())
    ])

    scoring = {
        'roc_auc': 'roc_auc',
        'log_loss': 'neg_log_loss',
        'accuracy': 'accuracy'
    }

    dummy_score_info = cross_validate(dummy_pipe,df,target,cv = StratifiedKFold(),scoring=scoring)
    lr_score_info = cross_validate(lr_pipe,df,target,cv = StratifiedKFold(),scoring=scoring)
    lr_better_roc_auc_count += int(np.mean(dummy_score_info['test_roc_auc']) < np.mean(lr_score_info['test_roc_auc']))
    lr_better_log_loss_count += int(np.mean(dummy_score_info['test_log_loss']) < np.mean(lr_score_info['test_log_loss']))
    lr_better_accuracy_count += int(np.mean(dummy_score_info['test_accuracy'] < np.mean(lr_score_info['test_accuracy'])))
print(f'Logistic regression had a better ROC-AUC score {lr_better_roc_auc_count} times out of 1000.')
print(f'Logistic regression had a better log-loss score {lr_better_log_loss_count} times out of 1000.')
print(f'Logistic regression had a better accuracy score {lr_better_accuracy_count} times out of 1000.')

Logistic regression had a better ROC-AUC score 534 times out of 1000.
Logistic regression had a better log-loss score 105 times out of 1000.
Logistic regression had a better accuracy score 556 times out of 1000.


I'm sure there's some reason that logistic regression has generally better ROC-AUC and accuracy scores while having far worse log-loss scores, but I'm not too worried about it.  In any case, the fact that logistic regression is worse here tells us that it's a bad model.

### Decision Trees
Here, we look at how effective decision trees are at learning rock-paper-scissors.

In [4]:
dt_pipe = Pipeline([
    ('one_hot',OneHotEncoder()),
    ('dt', DecisionTreeClassifier()) # I'm just using the default DecisionTreeClassifier since I don't really know what it does.
])

cv_info = cross_validate(dt_pipe,df,target,cv=StratifiedKFold(),scoring=scoring)
print("Decision Tree Classifier")
for scoring_key in scoring.keys():
    print("\t" + scoring_key + " mean:", np.mean(cv_info['test_' + scoring_key]))
    print("\t" + scoring_key + " std:", np.std(cv_info['test_' + scoring_key],ddof=1))

Decision Tree Classifier
	roc_auc mean: 1.0
	roc_auc std: 0.0
	log_loss mean: -2.2204460492503136e-16
	log_loss std: 0.0
	accuracy mean: 1.0
	accuracy std: 0.0


It seems like a decision tree learns rock-paper-scissors perfectly!  I suppose that means that the ensemble methods (gradient boosting and random forest models) probably will too.

## Models to Compare

- Baseline:
  - Baseline: Logistic regression on Elo differential.  It seems like higher Elo differentials (for p1) correspond to higher win rates (for p1), so this will probably be slightly better than a coin flip
  - Baseline: Random forest on Elo differential.  This is probably nonsense.
  - Baseline: Gradient boosting on Elo differential.  This is probably nonsense, but who knows?

- Without Elo differential (larger training pool):
  - Test model: Logistic regression on stats
  - Test model: Random forest on types/stats
  - Test model: Gradient boosting on types/stats.
  - Test model: Logistic regression on advantage data
  - Test model: Logistic regression on advantage + stats
  - Test model: Random forest on advantage data
  - Test model: Random forest on advantage + types + stats
  - Test model: Gradient boosting on advantage data
  - Test model: Gradient boosting on advantage + types + stats

- With Elo differential (smaller training pool):
  - Test model: Logistic regression on Elo differential + stats.  Will probably perform a bit better than baseline, but I don't anticipate by much because of the balancing.
  - Test model: Random forest on Elo differential + types/stats.  Will probably perform better than anything above this.
  - Test model: Gradient boosting on Elo differential + types/stats.  Don't know anything about this yet.
  - Same as above, but with Elo differential as a feature.

## Training data set to use

- Throw out any "incomplete" matches
  - Matches with duration under a certain threshold?
  - Matches which feature at least two pokemon revealed on each side?  (Probably not)
  - Matches which feature at least three pokemon revealed on one side?
- If training on Elo diff, keep only matches where both players' ratings are nonzero.

## Metrics to use

- Accuracy score seems pretty good, actually.  There's no meaningful difference between Type I and Type II errors; they're both just wrong predictions.
- Log-loss also seems great, at least for the models which give a probabilistic prediction.  That way, the model doesn't get punished for missing lots of close matches.
  - This is at least applicable to logistic regression and random forest (which is determined by voting, so it should have a predict_proba method floating around)
- AUC-ROC is useful in this situation due to its probabilistic interpretation, but probably not what we want to hinge our final decision on.
- Is there an F-score sort of situation with logistic regression?  Something where we can say 'training on advantage data + stats does not provide a meaningful improvement on advantage data alone?'

## Things worth looking at
- All models support confusion matrices.  Take a look to see if they are symmetric!  (This is desirable)
- Remember to see if we can spot any common features of matches where our predictions fail.
- The best cutoff probability is almost certainly 0.5.  There's no reason to privelege one player over the other.
  - Could we do a variable cutoff rate based on Elo differential?  Privelege the player with the higher Elo?
  - Due to best cutoff being 0.5, it doesn't make much sense to look at precision-recall (which thinks about precision and recall as a function of cutoff) or AUC-ROC (which thinks about TPR and FPR as a function of cutoff)
  - That said, AUC-ROC has a nice probabilistic interpretation: it yields the probability that for a randomly chosen pair of matches where p1 wins one and loses the other, the model rates the match where p1 won as more likely to be a p1 win than the loss.  So maybe it is okay to look at.
